# Day 09 — `asyncio.gather()`: Parallel Async Calls

**Phase 1 · Module 1: Sub-graphs** · ~30 min

### What I practice today
- [ ] See why `await`-ing calls **one after another** wastes time when they don't depend on each other
- [ ] Run them **at the same time** with **`asyncio.gather()`**
- [ ] **Time** sequential vs parallel and log the difference
- [ ] Know that `gather` **preserves input order** in its results (not finish order)
- [ ] Survive **partial failures** with `gather(..., return_exceptions=True)`
- [ ] Bridge: this is exactly how LangGraph runs **parallel nodes** — fan out, then merge (today's LangGraph notebook)


## 1. The everyday problem — three checks, done one at a time

Before a bank approves a card payment it runs a few independent checks: a **fraud score**, a **balance lookup**, a **sanctions/KYC check**. Each is a network call that takes a moment. Written the obvious way, you `await` them in a row — each one waits for the previous to *finish* before it even starts.

In [1]:
import asyncio, time

async def fraud_score(txn):
    await asyncio.sleep(0.3)      # network call to the fraud service
    return "fraud=0.02"

async def balance_check(txn):
    await asyncio.sleep(0.3)      # call to the ledger
    return "balance=OK"

async def kyc_check(txn):
    await asyncio.sleep(0.3)      # call to the sanctions list
    return "kyc=CLEAR"

async def approve_sequential(txn):
    t0 = time.perf_counter()
    a = await fraud_score(txn)    # wait 0.3s ...
    b = await balance_check(txn)  # ...THEN wait another 0.3s ...
    c = await kyc_check(txn)      # ...THEN another 0.3s
    dt = time.perf_counter() - t0
    return [a, b, c], dt

result, dt = await approve_sequential("txn-1")
print(result)
print(f"sequential took {dt:.2f}s")

['fraud=0.02', 'balance=OK', 'kyc=CLEAR']
sequential took 0.90s


☝ ~0.9s — the three delays **add up**, because each `await` fully blocks until its call returns before the next line runs. But these checks don't need each other: the fraud score doesn't depend on the balance. Making them wait in line is pure waste. They *could* all be in flight at once.

## 2. The fix — `asyncio.gather()` runs them concurrently

`asyncio.gather(coro1, coro2, ...)` schedules **all** the coroutines on the event loop at once and waits for **all** to finish. While one is parked on its `await asyncio.sleep` (i.e. waiting on the network), the others use that idle time. Total time ≈ the **slowest single call**, not the sum.

In [2]:
async def approve_parallel(txn):
    t0 = time.perf_counter()
    a, b, c = await asyncio.gather(     # all three start together
        fraud_score(txn),
        balance_check(txn),
        kyc_check(txn),
    )
    dt = time.perf_counter() - t0
    return [a, b, c], dt

result, dt = await approve_parallel("txn-1")
print(result)
print(f"parallel took {dt:.2f}s")

['fraud=0.02', 'balance=OK', 'kyc=CLEAR']
parallel took 0.30s


☝ ~0.3s instead of ~0.9s — a 3× win, and it grows with the number of calls. Note we passed the coroutines *without* `await` in front; `gather` awaits them. The three ran **overlapped**: all parked on their sleep at the same time, all woke together. This only helps for **I/O-bound** work (network, disk) — the waiting is what overlaps. CPU-crunching in a loop won't speed up this way.

## 3. Time them side by side — log the difference

The plan says *"compare sequential vs parallel timing — log the difference."* Run both and print the speed-up. (We'll log it structured, echoing Day 08's `structlog` habit — data, not a sentence.)

In [3]:
seq_result,  seq_dt  = await approve_sequential("txn-42")
par_result,  par_dt  = await approve_parallel("txn-42")

print(f"sequential : {seq_dt:.2f}s")
print(f"parallel   : {par_dt:.2f}s")
print(f"speed-up   : {seq_dt / par_dt:.1f}x faster")
print(f"time saved : {seq_dt - par_dt:.2f}s per payment")

sequential : 0.90s
parallel   : 0.30s
speed-up   : 3.0x faster
time saved : 0.60s per payment


☝ Same three checks, same results — only the *scheduling* changed. On one payment that's half a second; at a bank doing millions a day it's the difference between a snappy checkout and a spinner. The rule of thumb: **any time you `await` two things that don't depend on each other, you probably want `gather`.**

## 4. `gather` keeps **input order**, not finish order

A subtle, important guarantee: the results list matches the **order you passed the coroutines**, regardless of which actually finished first. So you can safely unpack `a, b, c` and know which is which — even if `c`'s service is fastest.

In [4]:
async def slow_then_label(label, delay):
    await asyncio.sleep(delay)
    return f"{label} (finished after {delay}s)"

# pass them fastest-last on purpose
results = await asyncio.gather(
    slow_then_label("first",  0.30),   # slowest
    slow_then_label("second", 0.10),
    slow_then_label("third",  0.01),   # fastest
)
for r in results:
    print(r)

first (finished after 0.3s)
second (finished after 0.1s)
third (finished after 0.01s)


☝ `third` finished first, but it's still **last** in the list — `gather` lines results up with the *inputs*. That's why you can write `a, b, c = await gather(A, B, C)` and trust the positions. (If you instead want results *as they complete*, that's `asyncio.as_completed` — a different tool.)

## 5. Partial failure — one bad call shouldn't sink the batch

By default, if **any** gathered coroutine raises, `gather` propagates that exception immediately and you lose the good results too. In a fan-out that's often wrong: if the KYC service is down you may still want the fraud and balance answers. Pass **`return_exceptions=True`** and failures come back **as values** in the list instead of blowing up.

In [5]:
async def flaky_kyc(txn):
    await asyncio.sleep(0.1)
    raise ConnectionError("kyc service timeout")

# default: one failure raises and hides the successes
try:
    await asyncio.gather(fraud_score("t"), flaky_kyc("t"))
except ConnectionError as e:
    print("default gather -> whole batch raised:", e)

# return_exceptions=True: failures become entries you can inspect
results = await asyncio.gather(
    fraud_score("t"), balance_check("t"), flaky_kyc("t"),
    return_exceptions=True,
)
for r in results:
    tag = "ERROR" if isinstance(r, Exception) else "ok"
    print(f"  [{tag:5}] {r!r}")

default gather -> whole batch raised: kyc service timeout
  [ok   ] 'fraud=0.02'
  [ok   ] 'balance=OK'
  [ERROR] ConnectionError('kyc service timeout')


☝ With `return_exceptions=True` the batch **always completes**; each slot is either a result or an `Exception` object you check with `isinstance`. Now you decide per-call what to do — proceed on the two that succeeded, retry the one that failed, or flag the payment for manual review. This is the difference between "one flaky service breaks everything" and graceful degradation.

## 6. Tie-in — this is LangGraph's parallel-node pattern

In today's LangGraph notebook you'll fan a graph out to **several nodes that run at once**, then a **merge** node combines their outputs. Under the hood LangGraph schedules those parallel nodes concurrently — the same idea you just built with `gather`: **independent work overlaps, then results are collected.**

Here's a mock of that fan-out → merge shape using pure `gather`, so the bridge is concrete:

In [6]:
async def research_node(source, topic):
    await asyncio.sleep(0.2)                 # each "node" hits a different source
    return f"{source}: 3 findings on {topic}"

async def parallel_graph(topic):
    # fan out: all research nodes run together
    findings = await asyncio.gather(
        research_node("news",    topic),
        research_node("filings", topic),
        research_node("social",  topic),
    )
    # merge: one node aggregates what the parallel branches produced
    summary = f"{len(findings)} sources merged -> " + " | ".join(findings)
    return summary

print(await parallel_graph("HSBC credit risk"))

3 sources merged -> news: 3 findings on HSBC credit risk | filings: 3 findings on HSBC credit risk | social: 3 findings on HSBC credit risk


☝ `research_node` × 3 in `gather` = LangGraph's **parallel branches**; the `summary` line = the **merge node**. LangGraph adds the graph machinery (state, reducers to collect the list, a `Send()` API for dynamic fan-out), but the performance idea is identical to what you just measured: **do the independent calls concurrently, then combine.**

## 7. Recap + checklist

| Tool | Why it was used here |
|------|----------------------|
| **`await` in a row** | simple, but independent calls run one-after-another — times **add up** |
| **`asyncio.gather(*coros)`** | run coroutines **concurrently**; total ≈ the slowest one |
| **input-order results** | `gather` returns results in the order passed, not finish order — safe to unpack |
| **`return_exceptions=True`** | failures come back as values, not a raised error — survive partial failure |
| **I/O-bound only** | overlap helps network/disk waits; CPU-bound loops won't speed up |

**Takeaway:** when you `await` several things that don't depend on each other, reach for `gather`. That's the exact move LangGraph makes when it runs **parallel nodes** and then a **merge** — today's LangGraph notebook.

> **No key / no install needed** — pure standard-library `asyncio`, runs fully offline.
